# Agricultural Expert System with LLM Enhancement
A domain-specific question-answering system designed to assist Egyptian farmers with crop management, disease diagnosis, and agricultural guidance.

The system is built on a manually curated knowledge base covering **90+ crops** with **60+ parameters per crop**, enhanced with a local LLM layer (Aya) to improve query understanding and response clarity for non-technical users.

In [8]:
import json
import os
import requests

## Loading Knowledge Base
Loading the manually curated expert data and question map from JSON files.
- **ExpertDatajson.json** — domain knowledge covering 90+ crops and 60+ parameters per crop, sourced and structured manually
- **questionmap.json** — maps common question patterns to data fields to improve query understanding

In [9]:
expert_data = json.loads(open('ExpertDatajson.json', encoding='utf-8').read())
question_map_data = json.loads(open('questionmap.json', encoding='utf-8').read())

## Configuration

In [10]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "aya" 

## Prompt bulding

In [11]:
def build_prompt(user_question, expert_knowledge, q_map=None):
    expert_knowledge_string = json.dumps(expert_knowledge, ensure_ascii=False, indent=2)

    q_map_section = ""
    if q_map:
        q_map_string = json.dumps(q_map, ensure_ascii=False, indent=2)
        q_map_section = f"""
دي بعض الأمثلة لطرق سؤال المستخدم عن البيانات، عشان تساعدك تفهم القصد حتى لو السؤال مش حرفي:
{q_map_string}
"""

    prompt = f"""
أنت شات بوت ذكي وخبير، بتتكلم عربي مصري متحضر ومحترف.
مهمتك الرئيسية هي الإجابة على أسئلة المستخدمين بدقة ومرونة بالاعتماد فقط على المعلومات المتاحة في البيانات اللي هقدمهالك.

افهم القصد: حاول تفهم المعنى الحقيقي لسؤال المستخدم حتى لو صياغة السؤال مختلفة.
جاوب من البيانات فقط: إجاباتك لازم تكون مستوحاة بشكل كامل من البيانات المتاحة ليك.
الاختصار والدقة: لو السؤال بيطلب معلومة محددة، جاوب على الجزء المطلوب بس بدون تفاصيل إضافية.
خارج النطاق: لو السؤال برا نطاق البيانات، قول إن المعلومة مش متوفرة بطريقة مهذبة.
اللغة: استخدم اللغة المصرية العامية المتحضرة.

البيانات المتاحة:
{expert_knowledge_string}
{q_map_section}
سؤال المستخدم: "{user_question}"
"""
    return prompt

## LLM Response Generator

In [12]:
def generate_response(prompt_text):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt_text,
            "stream": False,
            "num_predict": 200
        }
    )
    return response.json()['response']

## Questions function and Testing

In [13]:
def ask(question):
    prompt = build_prompt(question, expert_data, q_map=question_map_data)
    response = generate_response(prompt)
    print(f"السؤال: {question}")
    print(f"الإجابة: {response}\n")

In [14]:
ask("ازاي ازرع الطماطم وابيعها")

السؤال: ازاي ازرع الطماطم وابيعها
الإجابة: للإجابة على سؤالك، يمكننا النظر إلى عدة جوانب متعلقة بزراعة الطماطم وبيعها.

## زراعة الطماطم:
- اختيار نوع الطماطم المناسب: اختر أنواع الطماطم التي تتناسب مع مناخ منطقتك وذوق المستهلكين المحليين. بعض الأنواع الشائعة هي الطماطم الكرزية، والطماطم الرومي، والطماطم ذات النكهة القوية مثل الطماطم "بيف بوج".
- اختيار الوقت المناسب للزراعة: تبدأ فترة الزراعة المثالية عند بداية موسم الحرارة، عادة في منتصف الربيع إلى أوائل الصيف. تأكد من أن درجة الحرارة تتراوح بين 15-25 درجة مئوية أثناء نمو النبات.
- إعداد التربة: تحتاج الطماطم إلى تربة غنية بالمواد العضوية والمغذيات. قم بإعداد التربة عن طريق خلط السماد أو الكومبو العضوي مع التربة الموجودة. تأكد من أن التربة جيدة التصريف لمنع تعفن الجذور.
- زراعة البذور: يمكنك زراعة بذور الطماطم بعد آخر صقيع في الربيع. قم بزراعة البذور على عمق 2-3 سم مع ترك المسافة المناسبة بين كل بذرة، وعادة تكون حوالي 50-75 سم.
- رعاية النباتات: بعد الزراعة، تأكد من ري النباتات بانتظام والحفاظ على رطوبة التربة. قم بتسميد الطماطم مرة 

In [15]:
ask("ايه انسب وقت ازرع فيه الكرمب")

السؤال: ايه انسب وقت ازرع فيه الكرمب
الإجابة: بالتأكيد! فيما يلي إجابات على أسئلتك من القوائم المحددة:

- "الفترة المثالية لزراعة الكرمب": يفضل زراعة الكرمب خلال فصل الربيع، وتحديداً في أواخر شهر مارس أو أوائل شهر أبريل. فهذا الوقت يوفر للمحصول فترة نمو كافية قبل حلول الحرارة الشديدة في الصيف.

- "المدة بين الزراعة والحصاد": تختلف المدة حسب طرق الزراعة وظروف النمو، ولكن عادة ما يستغرق الكرمب حوالي 7-9 أشهر من وقت الزراعة حتى الحصاد. فبعد الزراعة مباشرة، ينمو النبات بسرعة ويبدأ في توليد السيقان والأوراق. ثم بعد ذلك، يتطور النبات إلى شكل الكرمش، ويستغرق الأمر حوالي شهرين إلى ثلاثة أشهر حتى يصبح ناضجاً للحصاد.

- "إنتاجية الفدان": يمكن أن تتراوح إنتاجية الفدان من الكرمب بين 5 إلى 10 أطنان، ولكن هذا يعتمد على عوامل مختلفة مثل نوع المحصول وظروف التربة والري والتسميد. فعلى سبيل المثال، قد ينتج نوع معين من الكرمب 7 أطنان للفدان، في حين أن نوعاً آخر قد ينتج 12 طناً.

- "التخزين السليم": للتخزين السليم للكرمب، يجب إتباع النصائح التالية:

   - يجب حصاد الكرمب عندما يكون ناضجاً تماماً، مع التأكد 

In [16]:
ask("ايه هي الامراض الي ممكن تقابل البطاطس")

السؤال: ايه هي الامراض الي ممكن تقابل البطاطس
الإجابة: بالتأكيد، يمكنني مساعدتك في ذلك. فيما يلي إجابات على أسئلتك بناءً على المعلومات المتوفرة في قاعدة البيانات:

- "ما هي الأمراض التي قد تصيب نبات البطاطس؟":
   - تعفن الجذور: ينتج عن تلف الجذور بسبب الفطريات أو البكتيريا، مما يؤدي إلى تعفنها وذبول النبات.
   - آفات الحشرات: يمكن أن تتعرض نباتات البطاطس لهجمات الحشرات مثل المن والعدوة والقراد.
   - أمراض فطرية: قد تصاب نباتات البطاطس بأمراض فطرية مثل البياض أو العفن الفطري.
   - نقص المغذيات: قد يؤدي نقص البوتاسيوم أو النيتروجين إلى ضعف نمو النبات وهشاشته للتلف.
   - الإصابة بالبكتيريا: يمكن أن تتعرض البطاطس لعدة بكتيريا مثل بكتيريا "الدراق" التي تسبب بقعًا بنية على الأوراق والجذور.

- "ما هي أعراض الإصابة بأمراض البطاطس؟":
   - التعفن الجذر: ظهور رائحة كريهة، وذبول النبات، وجفاف الأوراق.
   - آفات الحشرات: وجود ثقوب في أوراق النبات، وحشرات حية أو برازها على الأوراق والجذور.
   - الأمراض الفطرية: ظهور بقع بيضاء أو رمادية على الأوراق أو الجذور، وفقدان النمو الصحي.
   - نقص المغذيات: اص